# Get To Know A Dataset - Spatial Atlas of the Human Abdomen (SAHA)

This notebook serves as a guided tour of the [Spatial Atlas of the Human Abdomen (SAHA)](https://registry.opendata.aws/saha-open-data) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

**SAHA** is a multi-institutional, multi-platform spatial omics atlas of the human gastrointestinal and immune system. The atlas profiles ~15 million cells across 100+ donors using spatial transcriptomics platforms including CosMx, Xenium, GeoMx, G4X, and RNAscope. It covers organs such as the colon, ileum, liver, lung, lymph node, spleen, thymus, and appendix under both normal and disease conditions.

This notebook provides a **101-level introduction** to the SAHA dataset — how it is organized in S3, what data formats are present, and how to load and visualize the data.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The SAHA dataset is hosted in the `saha-open-data` S3 bucket and organized into four top-level prefixes:

1. **`metadata/`** — Four Parquet tables (`samples`, `donors`, `runs`, `panels`) that describe every sample, donor, acquisition run, and gene panel in the atlas. These are the starting point for any analysis.
2. **`raw/`** — Raw instrument output organized as `raw/{platform}/{institution}/{run_id}/{sample_id}/`. Platforms include `cosmx`, `xenium`, `geomx`, `g4x`, and `rnascope`.
3. **`processed/`** — Cell-level AnnData (`.h5ad`) and Seurat (`.rds`) objects with QC, normalization, cell typing, UMAP, and spatial coordinates pre-computed.
4. **`images/`** — Cloud-Optimized GeoTIFF and OME-Zarr tissue images for spatial visualization.

All metadata tables include an `s3_raw_prefix` and/or `s3_processed_path` column so you can jump directly from a query result to the underlying data files.

Full documentation: [QUICK_START.md](docs/QUICK_START.md) | [DATA_DICTIONARY.md](docs/DATA_DICTIONARY.md)

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38
# pandas >= 2.0
# pyarrow >= 14.0
# s3fs >= 2024.2
# matplotlib >= 3.8
# scanpy >= 1.10
# squidpy >= 1.4
# seaborn >= 0.13

import boto3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import squidpy as sq
import numpy as np
from botocore import UNSIGNED
from botocore.config import Config

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

S3_OPTS = {"anon": True}  # open data — no credentials needed

Next, we will connect to the S3 bucket and explore its top-level structure.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "saha-open-data"

# Create an unsigned (public) S3 client
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# List top-level prefixes
resp = s3.list_objects_v2(Bucket=bucket, Delimiter="/")
for item in resp.get("CommonPrefixes", []):
    print(item["Prefix"])

The `raw/` prefix is organized by platform, then institution, then run ID, then sample ID. Let's explore the CosMx data from Cornell as an example.

In [ ]:
# List platforms available under raw/
print("Platforms:")
for item in s3.list_objects_v2(Bucket=bucket, Prefix="raw/", Delimiter="/")["CommonPrefixes"]:
    print(f"  {item['Prefix']}")

print()

# List institutions under CosMx
print("CosMx institutions:")
for item in s3.list_objects_v2(Bucket=bucket, Prefix="raw/cosmx/", Delimiter="/")["CommonPrefixes"]:
    print(f"  {item['Prefix']}")

In [ ]:
# List files for one CosMx sample to show the raw data structure
prefix = "raw/cosmx/cornell/"
runs = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter="/")["CommonPrefixes"]
first_run = runs[0]["Prefix"]

# List samples in that run
sample_prefixes = s3.list_objects_v2(Bucket=bucket, Prefix=first_run, Delimiter="/")["CommonPrefixes"]
first_sample = sample_prefixes[0]["Prefix"]

print(f"Files in {first_sample}:")
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=first_sample, MaxKeys=15).get("Contents", []):
    size_mb = obj["Size"] / 1e6
    print(f"  {obj['Key'].split('/')[-1]:50s} {size_mb:>8.1f} MB")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

SAHA uses several data formats, each chosen for a specific purpose:

**Metadata (Parquet)**  
The four metadata tables (`samples`, `donors`, `runs`, `panels`) are stored as [Apache Parquet](https://parquet.apache.org/) files with Snappy compression. Parquet is a columnar format that enables fast filtering and is natively supported by Amazon Athena, pandas, R arrow, and DuckDB. The `samples` table is partitioned by `organ` and `platform` so that queries on these columns skip irrelevant files.

**Processed single-cell data (H5AD / RDS)**  
Each sample has a processed [AnnData](https://anndata.readthedocs.io/) object (`.h5ad`) containing the cell × gene count matrix, cell type annotations, spatial coordinates, and UMAP embeddings. Seurat `.rds` objects are also available for R users. H5AD was chosen because it integrates well with the `scanpy` / `squidpy` ecosystem for spatial omics.

**Raw instrument data (CSV/HDF5)**  
Raw data files are in the native format produced by each instrument:
- **CosMx**: gzipped CSVs (expression matrix, cell metadata, polygons) + TIFF label masks
- **Xenium**: HDF5 sparse matrix + CSV transcripts and cell boundaries + OME-TIFF images
- **GeoMx**: Excel/DCC count matrices + OME-TIFF scans
- **G4X**: Zarr image pyramids + Parquet cell tables

**Tissue images (Cloud-Optimized GeoTIFF / OME-Zarr)**  
Large morphology images are stored as Cloud-Optimized GeoTIFF or OME-Zarr to enable partial reads — you can load a thumbnail or a single FOV without downloading the entire image.

Recommended tools: Python (`pandas`, `scanpy`, `squidpy`, `spatialdata-io`), R (`Seurat`, `arrow`), AWS services ([Amazon Athena](https://docs.aws.amazon.com/athena/) for SQL queries, [S3 Select](https://docs.aws.amazon.com/AmazonS3/latest/userguide/selecting-content-from-objects.html) for targeted retrieval).

### Q: Can you show us an example of downloading and loading data from your dataset?

The best starting point is the **metadata tables** — they let you discover samples, then jump directly to the underlying data. Let's load all four metadata tables and explore.

In [ ]:
META_BASE = "s3://saha-open-data/metadata"

samples = pd.read_parquet(f"{META_BASE}/samples/", storage_options=S3_OPTS)
donors  = pd.read_parquet(f"{META_BASE}/donors/",  storage_options=S3_OPTS)
runs    = pd.read_parquet(f"{META_BASE}/runs/",    storage_options=S3_OPTS)
panels  = pd.read_parquet(f"{META_BASE}/panels/",  storage_options=S3_OPTS)

print(f"samples: {samples.shape[0]:,} rows × {samples.shape[1]} columns")
print(f"donors:  {donors.shape[0]:,} rows × {donors.shape[1]} columns")
print(f"runs:    {runs.shape[0]:,} rows × {runs.shape[1]} columns")
print(f"panels:  {panels.shape[0]:,} rows × {panels.shape[1]} columns")

In [ ]:
# Peek at the samples table
samples[["sample_id", "organ", "platform", "condition",
         "panel_plex", "n_cells", "n_fovs", "qc_pass"]].head(5)

In [ ]:
# Peek at the donors table
donors.head(5)

In [ ]:
# Panel configurations
for _, row in panels.iterrows():
    gene_count = len(row["gene_list"]) if row["gene_list"] is not None else 0
    print(f"{row['panel_name']:25s}  {row['platform']:8s}  plex={row['plex']:<6}  genes={gene_count}")

### Q: A picture is worth a thousand words. Show us visuals from your dataset that illustrate something informative or exciting.

Let's start with an overview of the atlas composition — how many samples and cells are available across organs, platforms, and conditions.

In [ ]:
# Filter to QC-passing samples for all overview plots
qc = samples[samples["qc_pass"] == True].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=100)

# --- Samples by organ ---
organ_counts = qc["organ"].value_counts().sort_values()
organ_counts.plot(kind="barh", ax=axes[0], color="#3498db", edgecolor="white")
axes[0].set_xlabel("Number of samples")
axes[0].set_title("Samples by organ", fontweight="bold")
axes[0].spines[["top", "right"]].set_visible(False)

# --- Total cells by platform ---
cells_by_platform = qc.groupby("platform")["n_cells"].sum().sort_values()
cells_by_platform.plot(kind="barh", ax=axes[1], color="#2ecc71", edgecolor="white")
axes[1].set_xlabel("Total cells")
axes[1].set_title("Total cells by platform", fontweight="bold")
axes[1].spines[["top", "right"]].set_visible(False)

# --- Condition breakdown ---
cond_counts = qc["condition"].value_counts()
axes[2].pie(cond_counts, labels=cond_counts.index, autopct="%1.0f%%",
            startangle=90, colors=sns.color_palette("Set2", len(cond_counts)))
axes[2].set_title("Condition distribution", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"\nTotal QC-passing samples: {len(qc):,}")
print(f"Total cells:             {qc['n_cells'].sum():,.0f}")
print(f"Unique donors:           {qc['donor_id'].nunique():,}")
print(f"Platforms:               {', '.join(sorted(qc['platform'].unique()))}")
print(f"Organs:                  {', '.join(sorted(qc['organ'].unique()))}")

In [ ]:
# Distribution of cell counts per sample, colored by platform
fig, ax = plt.subplots(figsize=(10, 5), dpi=100, facecolor="white")

for platform, group in qc.groupby("platform"):
    ax.hist(group["n_cells"], bins=40, alpha=0.6, label=platform, edgecolor="white")

ax.set_xlabel("Cells per sample", fontsize=12)
ax.set_ylabel("Number of samples", fontsize=12)
ax.set_title("Distribution of Cell Counts per Sample", fontsize=14, fontweight="bold")
ax.legend(title="Platform")
ax.spines[["top", "right"]].set_visible(False)
ax.set_facecolor("#f8f9fa")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Median cells/sample: {qc['n_cells'].median():,.0f}")
print(f"Max cells/sample:    {qc['n_cells'].max():,.0f}")

In [ ]:
# Heatmap: total cells per organ × platform
pivot = (
    qc.groupby(["organ", "platform"])["n_cells"]
    .sum()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
sns.heatmap(
    pivot, annot=True, fmt=",.0f", cmap="YlGnBu",
    linewidths=0.5, ax=ax, cbar_kws={"label": "Total cells"}
)
ax.set_title("Total Cells: Organ × Platform", fontsize=14, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("")

plt.tight_layout()
plt.show()

Now let's load a single processed sample and visualize the spatial cell type map — this is what makes SAHA a *spatial* atlas.

In [ ]:
# Pick a large, QC-passing colon sample on CosMx
candidates = qc.query(
    "organ == 'colon' and platform == 'cosmx' and panel_plex == 6000"
).sort_values("n_cells", ascending=False)

row = candidates.iloc[0]
print(f"Selected: {row['sample_id']}")
print(f"  organ:   {row['organ']}")
print(f"  n_cells: {row['n_cells']:,}")
print(f"  n_fovs:  {row['n_fovs']}")
print(f"  panel:   {row['panel_name']} ({row['panel_plex']}-plex)")
print(f"  S3 path: {row['s3_processed_path']}")

In [ ]:
# Load the processed AnnData object directly from S3
adata = sc.read_h5ad(
    row["s3_processed_path"],
    storage_options=S3_OPTS,
).to_memory()

print(adata)
print(f"\nCell metadata columns: {adata.obs.columns.tolist()}")
print(f"Embeddings available:  {list(adata.obsm.keys())}")

In [ ]:
# Spatial cell type map — the hallmark of spatial omics
if "spatial" in adata.obsm:
    fig, ax = plt.subplots(figsize=(12, 9), dpi=100)

    cell_types = (
        adata.obs["cell_type"].cat.categories
        if hasattr(adata.obs["cell_type"], "cat")
        else adata.obs["cell_type"].unique()
    )
    palette = sns.color_palette("tab20", len(cell_types))
    color_map = dict(zip(cell_types, palette))

    xy = adata.obsm["spatial"]
    colors = adata.obs["cell_type"].map(color_map)

    ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=1, alpha=0.6, linewidths=0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"{row['sample_id']} — cell types in tissue", fontsize=14, fontweight="bold")

    from matplotlib.patches import Patch
    handles = [Patch(color=color_map[ct], label=ct) for ct in cell_types]
    ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc="upper left",
              fontsize=7, frameon=False)

    plt.tight_layout()
    plt.show()
else:
    print("No spatial coordinates found in this sample.")

In [ ]:
# Spatial expression of marker genes
marker_genes = [g for g in ["EPCAM", "CD3E", "CD68", "MKI67", "COL1A1"]
                if g in adata.var_names]

if marker_genes and "spatial" in adata.obsm:
    sq.pl.spatial_scatter(
        adata,
        color=marker_genes,
        shape=None,
        size=2,
        img=False,
        ncols=len(marker_genes),
        figsize=(4 * len(marker_genes), 4),
    )
else:
    print(f"Available marker genes: {marker_genes}")

In [ ]:
# UMAP colored by cell type
if "X_umap" in adata.obsm:
    sc.pl.umap(
        adata,
        color="cell_type",
        legend_loc="on data",
        legend_fontsize=8,
        title=f"{row['sample_id']} — UMAP",
        frameon=False,
    )

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Question:** *How does cell type composition vary across GI organs, and do multi-organ donors show consistent immune representation?*

SAHA profiles multiple organs from the same donor, enabling direct within-donor comparisons of immune cell proportions. Below we compute cell type fractions per organ and look for patterns.

In [ ]:
# Aggregate cell counts by organ and platform
summary = (
    qc.groupby(["organ", "platform"])
    .agg(
        n_samples=("sample_id", "count"),
        total_cells=("n_cells", "sum"),
        n_donors=("donor_id", "nunique"),
    )
    .reset_index()
    .sort_values("total_cells", ascending=False)
)

print("Organ × Platform summary:")
summary.head(15)

In [ ]:
# Identify donors profiled across multiple organs
organs_per_donor = (
    qc.groupby("donor_id")["organ"]
    .nunique()
    .sort_values(ascending=False)
)

multi_organ_donors = organs_per_donor[organs_per_donor > 1]
print(f"Donors with multi-organ profiling: {len(multi_organ_donors)}")
print(f"\nTop donors by number of organs profiled:")
print(multi_organ_donors.head(10))

In [ ]:
# Join samples with donor demographics to see age and sex distributions
merged = qc.merge(
    donors[["donor_id", "sex", "age_group"]],
    on="donor_id", how="left",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=100)

# Cells by sex
merged.groupby("sex")["n_cells"].sum().plot(
    kind="bar", ax=axes[0], color=["#3498db", "#e74c3c"], edgecolor="white"
)
axes[0].set_title("Total cells by donor sex", fontweight="bold")
axes[0].set_ylabel("Total cells")
axes[0].spines[["top", "right"]].set_visible(False)

# Cells by age group
merged.groupby("age_group")["n_cells"].sum().sort_index().plot(
    kind="bar", ax=axes[1], color="#2ecc71", edgecolor="white"
)
axes[1].set_title("Total cells by donor age group", fontweight="bold")
axes[1].set_ylabel("Total cells")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations for someone wanting to answer this question?

**Question:** *Can cross-platform spatial transcriptomics data be integrated to build a unified cell atlas that is robust to technological batch effects?*

SAHA profiles the same organs using multiple spatial platforms (CosMx, Xenium, GeoMx, G4X). This creates a unique opportunity to benchmark integration methods and build a platform-agnostic reference atlas of the human abdomen. Key challenges include:

1. **Gene panel overlap**: Different platforms measure different gene sets (1K vs 6K vs 19K-plex). Integration must handle the partially-shared feature space.
2. **Resolution differences**: CosMx and Xenium provide single-cell resolution; GeoMx operates at the ROI level. Principled deconvolution or transfer learning is needed.
3. **Spatial alignment**: When the same tissue block is profiled on multiple platforms, spatial coordinates can be registered to enable direct pixel-level comparison.

**Recommendations:**
- Start with Tutorial 03 (`tutorials/03_cross_platform_comparison.ipynb`) which compares CosMx and Xenium on the same organ.
- Use `scvi-tools` or `scanorama` for integration, benchmarking against the shared gene set.
- The `panels` metadata table provides the full gene list per panel, which is essential for identifying the intersection.
- Consider the [Open Problems in Single-Cell Analysis](https://openproblems.bio/) benchmarking framework for rigorous evaluation.

---

## Further Resources

| Resource | Description |
|----------|-------------|
| [QUICK_START.md](docs/QUICK_START.md) | Concise access patterns (Python, R, AWS CLI, Athena) |
| [DATA_DICTIONARY.md](docs/DATA_DICTIONARY.md) | Full column definitions for all metadata tables |
| [FILE_FORMATS.md](docs/FILE_FORMATS.md) | Raw data formats per platform |
| [ACCESS_TIERS.md](docs/ACCESS_TIERS.md) | How to request registered/controlled-access data |
| Tutorial 01 | [Query metadata tables](tutorials/01_query_metadata.ipynb) |
| Tutorial 02 | [Load and explore a CosMx sample](tutorials/02_load_cosmx_sample.ipynb) |
| Tutorial 03 | [Cross-platform comparison](tutorials/03_cross_platform_comparison.ipynb) |

---
*Remember to clear all outputs before committing to your GitHub repository.*